[<img src="../quantumsymmetry_logo.png" alt="QuantumSymmetry" width="450"/>](https://github.com/dariopicozzi/quantumsymmetry)

> **Note:** if you are running this notebook on Google Colab, the next cell will install `quantumsymmetry` and its dependencies (this might take a few minutes):

In [1]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip -q install quantumsymmetry
    !pip -q install pylatexenc

# Dressing the ansatz: a symmetry-adapted Schrieffer-Wolff transformation

A **dressing** wraps the tree state in an ordered product of exact rotations

$$U(\boldsymbol\varphi) = \prod_k e^{-i\varphi_k A_k},$$

whose anti-Hermitian generators satisfy the cubic identity $A_k^3 = A_k$. That identity collapses each exponential to a closed form, so the dressing is applied exactly -- no Trotter error. One natural use is a **Schrieffer-Wolff transformation**: choosing $\boldsymbol\varphi$ so that the dressed Hamiltonian $\tilde H = U^\dagger H\,U$ decouples a low-energy *model space* $P$ from its complement $Q$. We do exactly that for a real molecule, $H_3^+$.

## The dressing is exact

Because $A^3 = A$, each rotation has the closed form $e^{-i\varphi A} = I + (\cos\varphi - 1)A^2 - i\sin\varphi\,A$. `make_dressing_pool` validates the cubic identity and caches $A^2$; `apply_dressing` then reproduces the matrix exponential to machine precision.

In [2]:
import numpy as np
from scipy.linalg import expm
from quantumsymmetry import make_dressing_pool, apply_dressing

# An off-block rotation generator i(|3><1| - |1><3|), which obeys A^3 = A.
A = np.zeros((8, 8), complex); A[3, 1], A[1, 3] = 1j, -1j
pool = make_dressing_pool([A])

rng = np.random.default_rng(0)
v = rng.standard_normal(8) + 1j * rng.standard_normal(8)
err = np.max(np.abs(apply_dressing(pool, np.array([0.7]), v) - expm(-0.7j * A) @ v))
print('apply_dressing vs expm :', f'{err:.1e}')

apply_dressing vs expm : 1.1e-16


## Symmetry-adapted generators for a molecule

For a chemistry problem the generators are the encoded single and double excitations that connect the model space $P$ to its complement $Q$. `dressing_generators` builds exactly these from a symmetry-adapted `Encoding` (the same object as the first tutorial series), already projected into the encoded qubits and screened to satisfy $A^3 = A$. We take $H_3^+$, encode it, and split its $2^3 = 8$ encoded states into the lower half $P$ and upper half $Q$.

In [3]:
from openfermion import get_sparse_operator
from quantumsymmetry import Encoding, dressing_generators, dressing_diagnostics

enc = Encoding(atom='H 0 0 0; H 0 0 1.0; H 0.94 0 0.5', basis='sto-3g', charge=1)
nq = enc.nspinorbital - len(enc.target_qubits)
dim = 1 << nq

# enc.hamiltonian is an OpenFermion QubitOperator; densify it on nq qubits.
H = get_sparse_operator(enc.hamiltonian, n_qubits=nq).toarray().astype(complex)

P, Q = list(range(dim // 2)), list(range(dim // 2, dim))
pool = make_dressing_pool(dressing_generators(enc, P, Q))

print('encoded qubits :', nq)
print('dressing pool  :', len(pool), 'generators')

encoded qubits : 3
dressing pool  : 6 generators


## Decoupling P from Q

We measure how strongly $P$ and $Q$ are coupled by the off-block **leakage** $\lVert Q\,\tilde H\,P\rVert_F^2 / |P|$, then minimise a Fubini-Study-sampled surrogate of it. The surrogate is fed model-space probe states drawn with `sample_sector_haar` from the previous notebook, and its (cost, gradient) oracle hands straight to any optimiser. Decoupling drives the leakage to zero, and the dressed model block then carries the true ground-state energy.

In [4]:
from scipy.optimize import minimize
from quantumsymmetry import (MinimalCircuit, decoupling_surrogate,
                             sample_sector_haar)

leak0 = dressing_diagnostics(H, P, Q, pool, np.zeros(len(pool)))['leakage']

# Fubini-Study probe states on the model space P.
mc = MinimalCircuit(nq, P, complex=True)
th, om = sample_sector_haar(nq, P, 2 * len(P), rng=np.random.default_rng(0))
chis = [mc.statevector(th[s], om[s]) for s in range(len(th))]

res = minimize(lambda p: decoupling_surrogate(H, pool, chis, p),
               np.zeros(len(pool)), jac=True, method='BFGS')
diag = dressing_diagnostics(H, P, Q, pool, res.x)

print('leakage before     :', f'{leak0:.2e}')
print('leakage after      :', f"{diag['leakage']:.2e}")
print('effective gs energy:', f"{diag['gs_energy']:.8f}")
print('exact gs (full H)  :', f'{float(np.linalg.eigvalsh(H)[0]):.8f}')

leakage before     : 1.68e-02
leakage after      : 7.78e-12
effective gs energy: -1.27158962
exact gs (full H)  : -1.27158962


The leakage falls by ten orders of magnitude and the decoupled model block reproduces the exact ground-state energy: the dressing has rotated the low-energy physics entirely into $P$. The same recipe -- exact $A^3 = A$ rotations, symmetry-adapted generators, a Fubini-Study-sampled decoupling surrogate -- is what the paper applies to larger active spaces.

### The series

1. <a href="tree_01_welcome.ipynb" />Welcome: the binary-tree ansatz</a>
2. <a href="tree_02_ansatz_and_metric.ipynb" />The ansatz, its pruning and its diagonal metric</a>
3. <a href="tree_03_vqe.ipynb" />Natural-gradient VQE for molecules</a>
4. <a href="tree_04_dynamics.ipynb" />Real-time evolution</a>
5. <a href="tree_05_sampling.ipynb" />Sector-Haar sampling</a>
6. <a href="tree_06_dressing.ipynb" />Dressing the ansatz: a symmetry-adapted Schrieffer-Wolff transformation</a>
7. <a href="tree_07_spin.ipynb" />Spin adaptation: exact total spin on the tree</a>

<p style="text-align: left"> <a href="tree_05_sampling.ipynb" />&lt; Previous: Sector-Haar sampling</a> </p>
<p style="text-align: right"> <a href="tree_07_spin.ipynb" />Next: Spin adaptation: exact total spin on the tree &gt;</a> </p>